# Plot Back-slip inversion of the GNSS displacement measurements of interseismic locking (coupling) at Nicoya (Costa Rica) within a heterogeneous half-space, compared with a homogeneous solution



In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/real/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
# flag_savefig = True
flag_savefig = False

# Define the inversion results path
resultpath = "rst_locking/"

# define mesh name
# meshname = "nicoya"  # larger fault interface
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshname = "nicoyaCK2"   # same as above but 5-km mesh size on fault
meshname = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshname = "nicoyaCK4"   # same as CK2, but connecting the trench now
print(meshname)

# # Read the plate interface file
# plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
print(trench.head())

# define the type of disp. data to use
type = "both"   # use both trench parallel and normal components
# type = "dip"    # use only trench normal components     

# read in the lon and lat of stations
# obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

# note that the height is in m, Dt in years, the original displacement data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                 names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                        'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                        'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive
# Convert mm to m, needed for inversion
cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
              'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
df[cols_to_convert] = df[cols_to_convert] / 1e3  # Convert mm to m

# displacement noise standard deviations, in m 
error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())


In [ ]:
# flag to indicate if the slip inversion was bounded
BOUNDED = True
# BOUNDED = False

BOUND_TYPE = 'both'

# choose function type,  available choices are 'tanh'- Hyperbolic tangent (default), 'arctan' - Arctangent scaled by 2/π  
# 'sigmoid' - 2/(1+exp(-x)) - 1, and 'sqrt' - x/sqrt(1+x²)
FUNCTION_TYPE = 'tanh'
# FUNCTION_TYPE = 'sigmoid'

if BOUNDED:
    # Define slip bounds based on your problem
    V_para = 27.0 / 1e3     # the max trench-parallel long-term loading of 27 mm
    v_const_para = 11.0 / 1e3  # a constant value from trench parallel component 
    s_strike_max = V_para - v_const_para / 1e3    # remove non-elastic motion
    V_norm = 78.5 / 1e3     # the trench-normal long-term loading of 78.5 mm
    s_dip_max = V_norm
    if BOUND_TYPE == 'both':
        # strike_bounds=(0.0, s_strike_max)
        strike_bounds=(-V_para, V_para),
        dip_bounds=(0.0, s_dip_max)
        print("Constraints to both strike and dip ")

In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
print(volcano.head())

# region=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 
volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]


In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1
# f_h, f_v = 1, 1/5

# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# the optimal weight combination comes from the L-curve test
# rho_s = 1e7
rho_s = 1e8
# rho_s = 1e9

if BOUNDED:
    # gamma_val_H1 = 2e1
    gamma_val_H1 = 5e1  # seems the best 
    # gamma_val_H1 = 1e2
    # gamma_val_H1 = 2e2
    # gamma_val_H1 = 4e2
    # gamma_val_H1 = 5e2
    # gamma_val_H1 = 8e2
    # gamma_val_H1 = 1e3

else:
    gamma_val_H1 = 1e2
    # gamma_val_H1 = 1e3
    # gamma_val_H1 = 2e3
    # gamma_val_H1 = 5e3

delta_val_L2 = gamma_val_H1 / rho_s

gamma_val_H1 = 5e1  
delta_val_L2 = 5e-7
# gamma_val_H1 = 1e2  
# delta_val_L2 = 2e-6
# gamma_val_H1 = 2.5e3  
# delta_val_L2 = 1e-5

if BOUNDED:
    if FUNCTION_TYPE == 'tanh':
        inv_str = f"_lockbothbd_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        # inv_str = f"_robinlockbothbd_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        # inv_str = f"_steeplockbothbd_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        # inv_str = f"_m0lockbothbd_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
    else:
        inv_str = f"_lockbothbd{FUNCTION_TYPE[:4]}_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        
else:
    inv_str = f"_lockboth_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
    # inv_str = f"_lockingboth_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
print(inv_str)


In [ ]:
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot

    return x_rot, y_rot

# Load the ground-truth displacement for each forward structure model
def read_obs_disp(datadir, resultpath, meshname, inv_str):
    outFileName = 'd_obs_' + meshname + inv_str + '.txt'
    print(outFileName)
    u_obs = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    # the saved obs disp are aligned with mesh, reverse rotation, back to geo
    u_obs['ux'], u_obs['uy'] = rot_xy(u_obs['ux'], u_obs['uy'], -rot) 
    # print(u_obs.head())

    return u_obs

# A different way of constructing the vectors for plotting arrows
def build_disp_vector(df, u_obs, error_e, error_n, error_v):
    # convert from m to mm, horizontal components
    df_obs_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_obs['ux']*1000,
            "north_velocity": u_obs['uy']*1000,
            "east_sigma": error_e*1000,
            "north_sigma": error_n*1000,
            "correlation_EN": 0.0,
        } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
    )
    # print(df_obs_h.head())
    # np.ptp(np.sqrt(df_obs_h['east_velocity']**2 + df_obs_h['north_velocity']**2))

    # convert from m to mm, vertical components
    df_obs_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_obs['uz']*1000,
            "east_sigma": error_v*1000,
            "north_sigma": error_v*1000,
            "correlation_EN": 0.0,
        } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
    )
    
    return df_obs_h, df_obs_v

In [ ]:
# read in the observation disp.
u_obs = read_obs_disp(datadir, resultpath, meshname, inv_str)

# buil observation disp. vectors
df_obs_h, df_obs_v = build_disp_vector(df, u_obs, error_e, error_n, error_v)


In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
# mu_l = -0.9730  # ~25 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = 0.9730 # ~55 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f" %(mu_upper, mu_lower) )
    # string for the contrast between slab and wedge
    slabwedge_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # string for the 3d model
    het3d_str = "_DeShon3D"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    # homo_str = f"_mul{round(mu_expression(-0.9730))}u{round(mu_expression(-0.9730))}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    anomaly_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(slabwedge_str)
print(het3d_str)

# # define the structure model strings for the INVERSION 
# struc_str_inv_het = anomaly_str
# print("Inverse problem based on: ", struc_str_inv_het)

# struc_str_inv_hom = homo_str
# print("Inverse problem based on: ", struc_str_inv_hom)



In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
# offset in x and y direction, the same as being done to the mesh in 'Kyriakopoulos2016JGR/convert_exodus_to_msh.ipynb'
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m
km2m = 1e3  # conversion factor from km to m

# Load the fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f/km2m

# Compute the inverse projection
loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)
print(loc_f[['lon', 'lat']].head())
# fault.head()


In [ ]:
# Load the predicted surface displacement, all in meters
def read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_pred['ux'], u_pred['uy'] = rot_xy(u_pred['ux'], u_pred['uy'], -rot) 
    
    u_res = u_pred.copy()
    u_res['ux'] = u_obs['ux'] - u_pred['ux']
    u_res['uy'] = u_obs['uy'] - u_pred['uy']
    u_res['uz'] = u_obs['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    # u_res.head()

    return u_pred, u_res

# Load the inferred slip on the fault, all in meters
def read_inferred_slip(datadir, resultpath, meshname, inv_str, struc_str_inv, type):
    outFileName = 'm_s_fault_' + meshname + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    if type == 'both':
        m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['s_strike', 's_dip'])
        m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)

        cols_to_convert = ['s_strike', 's_dip', 'mag']
        m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm
        print(m_s['s_strike'].min(), m_s['s_strike'].max())
        print(m_s['s_dip'].min(), m_s['s_dip'].max())
        # print(m_s['mag'].max())

    elif type == 'dip':
        m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['s_dip'])
        cols_to_convert = ['s_dip']
        m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm
        m_s['mag'] = m_s['s_dip']
        print(m_s['s_dip'].min(), m_s['s_dip'].max())
        print(m_s['mag'].max())

    return m_s

In [ ]:
##### Load the results of the homogeneous inversion
# Load the predicted surface displacement, all in meters
u_pred_hom, u_res_hom = read_pred_disp(u_obs, datadir, resultpath, meshname, inv_str, homo_str)
# Load the inferred slip on the fault, all in meters
m_s_hom = read_inferred_slip(datadir, resultpath, meshname, inv_str, homo_str, type)

In [ ]:
# define interseismic coupling between the two plates as the ratio of back normal slip rate
# (m_est vector) to local trench-normal convergence rate (m_est/V_norm.)
V_norm = 78.5   # trench-normal of Cocos-Caribbean motion, mm/yr
V_const_para = 11     # only remove the a constant value from trench parallel component, mm/yr 
V_para = 27-V_const_para    # trench-parallel of Cocos-Caribbean motion, mm/yr   

def interseismic_coupling(m_s, V_norm):
    m_s['locking'] = m_s['s_dip']/V_norm
    m_s['locking1'] = m_s['mag']/V_norm
    m_s['locking2'] = 1-m_s['mag']/V_norm

    return m_s

m_s_hom = interseismic_coupling(m_s_hom, V_norm)

print(m_s_hom['locking'].min(), m_s_hom['locking'].max(), m_s_hom['locking'].median())


In [ ]:
indices = m_s_hom.index[(m_s_hom['locking'] >= 0.475) & (m_s_hom['locking'] <= 0.525)].tolist()
# print(indices)
m_s_hom_subset = m_s_hom.loc[indices].copy()

In [ ]:
def plot_all_inferred_slip_(m_s_hom, m_s_sw, m_s_3d, col2plt, region, flag_savefig=False):

    slipsybsz = "c0.05c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "rainbow"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.5", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=False)    #m_s_hom[col2plt].max()
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")
            # fig.plot(x=loc_f.loc[m_s_hom_subset.index, 'lon'], y=loc_f.loc[m_s_hom_subset.index, 'lat'],
            #     style="c0.05c",      # circle symbol with diameter 0.2 cm
            #     pen="0.5p,black",     # outline: 1-point wide, black color
            #     fill="-"         # no fill → open circles
            # )
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInterseismic Coupling"]) #, "y+l(mm)"

        # # slip inferred from the slab/wedge model
        # with fig.set_panel(panel=[0, 1]):
        #     fig.basemap(region=region, projection="M?", frame=["a1f0.5", "+tS/W inv."])
        #     # maxslip = m_s_sw[col2plt].max()
        #     pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
        #     grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
        #     # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
        #     # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
        #     # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
        #     # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

        #     fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
        #     # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
        #     # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

        #     fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        #     grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
        #     fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
        #     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        #     # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #     with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #         fig.colorbar(frame=["af", "x+lInterseismic Coupling"])

        # # slip inferred from the 3d model
        # with fig.set_panel(panel=[0, 2]):
        #     fig.basemap(region=region, projection="M?", frame=["a1f0.5", "+t3D inv."])
        #     # maxslip = m_s_3d[col2plt].max()
        #     pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=False)    #m_s_hom[col2plt].max()
        #     grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
        #     # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
        #     # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
        #     # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
        #     # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

        #     fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
        #     # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
        #     # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,black")

        #     fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        #     grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
        #     fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
        #     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        #     # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #     with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
        #         fig.colorbar(frame=["af", "x+lInterseismic Coupling"])

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + "_sum_invslip.pdf")


plot_all_inferred_slip_(m_s_hom, m_s_hom, m_s_hom, 'locking', region, flag_savefig=False)
# plot_all_inferred_slip_(m_s_hom, m_s_hom, m_s_hom, 'locking1', region, flag_savefig=False)


In [ ]:
def build_pred_disp_vector(df, u_pred, u_res, error_e, error_n, error_v):
    ### data fitting by the heterogeneous inversion 
    # convert from m to mm
    df_pred_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_pred['ux']*1000,
            "north_velocity": u_pred['uy']*1000,        
        }
    )
    df_pred_h.head()

    df_pred_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_pred['uz']*1000,        
        }
    )

    # convert from m to mm, in order to directly compare with 
    df_res_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_res['ux']*1000,
            "north_velocity": u_res['uy']*1000,
            "east_sigma": error_e*1000,
            "north_sigma": error_n*1000,
            "correlation_EN": 0.0,        
        }
    )
    df_res_h.head()
    df_res_h_dum = df_res_h.iloc[:, :4]

    df_res_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_res['uz']*1000,   
            "east_sigma": error_v*1000,
            "north_sigma": error_v*1000,
            "correlation_EN": 0.0,     
        }
    )
    df_res_v_dum = df_res_v.iloc[:, :4]

    return df_pred_h, df_pred_v, df_res_h, df_res_v, df_res_h_dum, df_res_v_dum


# buil observation disp. vectors for various structure models
df_pred_h_hom, df_pred_v_hom, df_res_h_hom, df_res_v_hom, \
    df_res_h_dum_hom, df_res_v_dum_hom = build_pred_disp_vector(df, u_pred_hom, u_res_hom, error_e, error_n, error_v)


In [ ]:
def plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h, df_res_h, df_res_h_dum, 
                               u_true, u_pred, color, lgdstr, 
                               region, struc_str_inv, errorbar=True, flag_savefig=False):
    # plot the horizontal predicted displacements vs. observed at GNSS stations, and the residual
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,red", uncertaintyfill=None, line="0.25p,red", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,red+ea+gred")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_h, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.05/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.5/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Pred {lgdstr}", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Residual  
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            if errorbar:
                fig.velo(data=df_res_h, pen=f"0.5p,{color}", uncertaintyfill=None, line=f"0.25p,{color}", 
                        spec="e0.16/0.39", vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            else:
                fig.velo(data=df_res_h_dum, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.16/0.39", 
                        vector=f"0.1c+a60+p1p,{color}+ea+g{color}")       
            rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm   
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.25, font="9p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1]})
            if errorbar:
                lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                    "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            else:
                lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res {lgdstr}", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            if errorbar:
                fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            else:
                fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")

    fig.show()
    
    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_datafit_h.pdf")


plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h_hom, df_res_h_hom, df_res_h_dum_hom, 
                           u_obs, u_pred_hom, 'black', 'hom', region, homo_str, errorbar=True, flag_savefig=False)


In [ ]:
def plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v, df_res_v, df_res_v_dum, u_true, u_pred, 
                                color, lgdstr, region, struc_str_inv, errorbar=True, flag_savefig=False):
    # plot the vertical predicted displacements vs. observed at GNSS stations, and the residual
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,red", uncertaintyfill=None, line="0.25p,red", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,red+ea+red")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_v, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.05/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.5/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Pred {lgdstr}", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Residual  
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="af", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            if errorbar:
                fig.velo(data=df_res_v, pen=f"0.5p,{color}", uncertaintyfill=None, line=f"0.25p,{color}", 
                        spec="e0.16/0.39", vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            else:
                fig.velo(data=df_res_v_dum, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.16/0.39", 
                        vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm   
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.25, font="9p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            if errorbar:
                lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            else:
                lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res {lgdstr}", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            if errorbar:
                fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.3, font="8p,Helvetica,black", justify="ML")
            else:
                fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.3, font="8p,Helvetica,black", justify="ML")

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_inv + "_datafit_v.pdf")


plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v_hom, df_res_v_hom, df_res_v_dum_hom, 
                            u_obs, u_pred_hom, 'black', 'hom', region, homo_str, errorbar=True, flag_savefig=False)
